# 30. MS 6x6 고정 embedding QA 실행

## 배경

본 실험(notebook 05)에서 MS 6x6은 timeout이 부족해 embedding에 실패했고 `NOT_EMBEDDABLE`로 기록되었습니다. 이후 embedding 실험(notebook 14)에서 **Pegasus, tries=10, timeout=2400** 조건으로 5개 seed 전부 성공했습니다.

| seed | 물리 큐빗 | 평균 chain | 최대 chain |
|---|---|---|---|
| 2024 | 4,503 | 17.39 | 34 |
| 2026 | 4,537 | 17.52 | **31** |
| 2025 | 4,675 | 18.05 | 35 |
| 2028 | 4,705 | 18.17 | 40 |
| 2027 | 4,755 | 18.36 | 37 |

## 이 notebook의 목적

1. notebook 05에서 비어 있던 MS 6x6 칸을 채웁니다.
2. **chain break의 기여와 coefficient range의 기여를 분리**합니다.

두 번째가 핵심입니다. MS 4x4는 embedding도 되고 chain break도 0.025%로 매우 낮았는데 feasible 해가 **0개**였습니다. 그래서 원인을 coefficient range로 추정했는데, 확증은 아니었습니다.

6x6은 최대 chain이 31~40으로 4x4(최대 14~17)보다 훨씬 깁니다. 두 경우를 나란히 놓으면 어느 요인이 지배적인지 가늠할 수 있습니다.

| 관측 | 해석 |
|---|---|
| 6x6도 chain break가 낮은데 feasible 0 | coefficient range가 지배적 |
| 6x6에서 chain break가 급증 | chain 길이도 유의한 요인 |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.config import load_config, resolve_path

config = load_config(PROJECT_ROOT / "config" / "experiment_config.yaml")
DATA_DIR = resolve_path(config, "data_dir")
RAW_DIR = resolve_path(config, "raw_dir")
PROCESSED_DIR = resolve_path(config, "processed_dir")
FIGURE_DIR = resolve_path(config, "figure_dir")
EMBEDDING_DIR = RAW_DIR / "embeddings"
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("설정 로드 완료")


설정 로드 완료


## 설정

`CHAIN_STRENGTH_ALPHA`는 본 실험의 고정값 **1.5**입니다. `chain_strength = alpha * max|J|` 규칙의 alpha이고, 이 값을 바꾸는 것은 parameter tuning에 해당하므로 **먼저 1.5로 한 번 돌린 뒤** 결과를 보고 판단하십시오. 바꿔서 돌린 결과는 본 실험이 아니라 진단 실험으로 분류해 기록해야 합니다.

`SELECT_METRICS`는 어떤 지표로 seed를 고를지입니다. 세 지표가 서로 다른 seed를 가리킬 수 있으므로(2024는 큐빗 최소, 2026은 최대 chain 최소), 각 지표의 최선을 모두 뽑은 뒤 중복을 제거합니다.

In [ ]:
TARGET_INSTANCE = "6x6"
TARGET_FORMULATION = "MS"
TARGET_TOPOLOGY = "pegasus"

# embedding 탐색 조건 (notebook 14에서 성공했던 조건과 동일해야 함)
EMBED_TRIES = 10
EMBED_TIMEOUT = 2400

# seed 선택
SELECT_METRICS = ("physical_qubits", "avg_chain_length", "max_chain_length")
MAX_SEEDS = None          # None이면 선정된 전부

# QA 파라미터. alpha는 본 실험 고정값 1.5로 먼저 돌린다.
CHAIN_STRENGTH_ALPHA = 1.5
NUM_READS = 2000          # None이면 config의 qa.num_reads
ANNEALING_TIME = 200     # None이면 config의 qa.annealing_time

print(f"대상      : {TARGET_INSTANCE} {TARGET_FORMULATION} / {TARGET_TOPOLOGY}")
print(f"embedding : tries={EMBED_TRIES}, timeout={EMBED_TIMEOUT}s")
print(f"alpha     : {CHAIN_STRENGTH_ALPHA}  (본 실험 고정값)")
print(f"num_reads : {NUM_READS or config['qa']['num_reads']}")
print(f"annealing_time : {ANNEALING_TIME or config['qa']['annealing_time']}")

대상      : 6x6 MS / pegasus
embedding : tries=10, timeout=2400s
alpha     : 1.5  (본 실험 고정값)
num_reads : 2000
annealing_time : 150


## seed 선택

embedding 실험 결과에서 품질이 좋은 seed를 고릅니다. 물리 큐빗이 가장 적은 embedding과 최대 chain이 가장 짧은 embedding이 다를 때 어느 쪽이 QA에 유리한지는 미리 알 수 없으므로, **둘 다 돌려서 비교**합니다.

이것 자체가 "embedding 품질이 QA 결과에 영향을 주는가"라는 별도 질문에 대한 대조 실험이 됩니다.

In [3]:
from src import fixed_embedding_qa as FQ
from src.persistence import load_table

embedding_results = load_table(PROCESSED_DIR, "embedding_study.csv")
selected = FQ.select_seeds(
    embedding_results,
    instance=TARGET_INSTANCE,
    formulation=TARGET_FORMULATION,
    topology=TARGET_TOPOLOGY,
    metrics=SELECT_METRICS,
    max_seeds=MAX_SEEDS,
)
selected[[
    "seed", "selected_by", "physical_qubits",
    "avg_chain_length", "max_chain_length", "tries", "timeout",
]]

,seed,selected_by,physical_qubits,avg_chain_length,max_chain_length,tries,timeout
0,2024,"physical_qubits, avg_chain_length",4503.0,17.386100,34.0,10,2400
1,2026,max_chain_length,4537.0,17.517375,31.0,10,2400


## embedding 확보

mapping이 JSON으로 저장돼 있으면 재사용하고, 없으면 탐색해서 저장합니다.

**저장본을 쓰기 전에 현재 working graph에서 유효한지 검사합니다.** QPU의 working graph는 교정 과정에서 바뀔 수 있고, 그러면 chain에 쓰인 물리 큐빗이 사라지거나 연결이 끊겨 mapping을 쓸 수 없게 됩니다. 이 경우 자동으로 재탐색합니다.

재탐색은 seed당 수백~1,400초가 걸릴 수 있습니다.

In [5]:
from src.data_generator import CFLPInstance

instance = CFLPInstance.load(DATA_DIR / f"{TARGET_INSTANCE}.json")

embeddings = {}
for _, row in selected.iterrows():
    seed = int(row["seed"])
    print(f"--- seed {seed} ({row['selected_by']}) ---")
    embedding, meta = FQ.obtain_embedding(
        instance=instance,
        formulation=TARGET_FORMULATION,
        config=config,
        seed=seed,
        tries=EMBED_TRIES,
        timeout=EMBED_TIMEOUT,
        topology=TARGET_TOPOLOGY,
        embedding_dir=EMBEDDING_DIR,
    )
    embeddings[seed] = (embedding, meta)
print(f"\n{len(embeddings)}개 embedding 확보")

--- seed 2024 (physical_qubits, avg_chain_length) ---
저장된 embedding 재사용: 6x6_MS_qpu-pegasus_seed2024_tries10.json
--- seed 2026 (max_chain_length) ---
저장본이 현재 그래프에서 유효하지 않습니다 (사라진 큐빗 21, 끊긴 chain 19, 사라진 연결 23). 재탐색합니다.
embedding 탐색 중 (seed=2026, tries=10, timeout=2400s)...
저장: 6x6_MS_qpu-pegasus_seed2026_tries10.json  큐빗 4537, 최대 chain 31

2개 embedding 확보


## QUBO 생성

본 실험과 동일한 설정으로 만듭니다. penalty도 기본값 그대로입니다.

In [6]:
from src.qubo_builder import build_qubo, qubo_statistics

model = build_qubo(
    instance,
    TARGET_FORMULATION,
    float(config["penalty"]["margin"]),
    int(config["encoding"]["precision"]),
)
stats = qubo_statistics(model)
print(f"변수 {stats['qubo_variables']}, 이차항 {stats['qubo_quadratic_terms']}")
print(f"lambda {stats['penalty_lambda']:.4e}")
print(f"계수 범위 {stats['qubo_range']:.4e}")

gurobi = pd.read_csv(RAW_DIR / "gurobi_results.csv")
reference = float(
    gurobi[(gurobi["instance"] == TARGET_INSTANCE)
           & (gurobi["gurobi_model"] == TARGET_FORMULATION)]["objective"].iloc[0]
)
print(f"Gurobi optimum {reference:.2f}")

변수 259, 이차항 8701
lambda 1.2971e+04
계수 범위 1.1903e+08
Gurobi optimum 3968.85


## QA 실행

선정된 embedding 각각에 대해 QA를 돌립니다. QUBO와 alpha는 동일하고 **embedding만 다릅니다.** 따라서 결과 차이는 embedding 품질에서 옵니다.

In [48]:
records = []
for seed, (embedding, meta) in embeddings.items():
    print(f"--- seed {seed} QA 실행 ---")
    outcome = FQ.run_qa_with_embedding(
        model=model,
        instance=instance,
        embedding=embedding,
        config=config,
        chain_strength_alpha=CHAIN_STRENGTH_ALPHA,
        solver_id=meta.get("solver_id"),
        num_reads=NUM_READS,
        annealing_time=ANNEALING_TIME,
    )
    row = FQ.to_schema_row(
        outcome=outcome,
        meta=meta,
        model=model,
        instance=instance,
        formulation=TARGET_FORMULATION,
        selected_by=selected[selected["seed"] == seed]["selected_by"].iloc[0],
        include_linking=False,
    )
    records.append(row)
    print(
        f"  chain break 평균 {row['qa_chain_break_fraction']:.4f}  "
        f"feasible {row['feasible_fraction'] * 100:.2f}%  "
        f"best feasible {row['best_feasible_objective']}"
    )

results = FQ.to_schema_frame(records)
print(f"\n{len(results)} 행, 컬럼 {len(results.columns)}개")

--- seed 2024 QA 실행 ---
  chain break 평균 0.0141  feasible 0.00%  best feasible None
--- seed 2026 QA 실행 ---
  chain break 평균 0.0140  feasible 0.00%  best feasible None

2 행, 컬럼 42개


## 결과 저장

**`qa_results.csv`(notebook 05)는 건드리지 않고 별도 파일 `qa_results_fixed_embedding.csv`에 저장합니다.** 컬럼 구성과 순서는 `qa_results.csv`와 동일하게 맞췄으므로, 나중에 두 파일을 그대로 concat 해서 비교할 수 있습니다.

다만 스키마에 없는 컬럼 세 개를 **뒤에 덧붙였습니다.**

| 컬럼 | 이유 |
|---|---|
| `seed` | 같은 instance를 여러 embedding으로 돌리므로, 없으면 행을 구분할 수 없습니다 |
| `selected_by` | 그 seed가 어떤 지표로 뽑혔는지 |
| `solver_id` | embedding이 어느 working graph 기준인지 |

이 세 개가 불필요하면 `results.drop(columns=FQ.EXTRA_COLUMNS)`로 빼시면 됩니다. 다만 `seed`를 빼면 두 행이 물리 큐빗 수로만 구분되니 권하지 않습니다.

alpha를 바꿔 다시 돌리면 같은 파일에 누적되며, `(instance, formulation, seed, qa_chain_strength_alpha)` 조합으로 구분됩니다.

In [49]:
from src.persistence import save_table

OUTPUT = PROCESSED_DIR / "qa_results_fixed_embedding.csv"

key = [
    "instance",
    "formulation",
    "seed",
    "solver_id",
    "qa_chain_strength_alpha",
    "qa_annealing_time",
    "num_reads",
]

if OUTPUT.exists() and OUTPUT.stat().st_size > 0:
    previous = pd.read_csv(OUTPUT)
    merged = pd.concat([previous, results], ignore_index=True)
    merged = merged.drop_duplicates(subset=key, keep="last")
    merged = FQ.to_schema_frame(merged.to_dict("records"))
else:
    merged = results.copy()

save_table(
    merged,
    PROCESSED_DIR,
    "qa_results_fixed_embedding.csv",
)

print(f"저장: {OUTPUT} ({len(merged)}행)")
print("컬럼:", ", ".join(merged.columns))

display(
    merged[[
        "seed",
        "num_reads",
        "qa_annealing_time",
        "qa_chain_strength_alpha",
        "feasible_fraction",
        "qa_chain_break_fraction",
        "runtime",
        "qa_qpu_access_time_us",
        "qa_qpu_sampling_time_us"
    ]]
)

저장: C:\Users\User\Desktop\KMJ\Study\Quantum\cflp_formulation\results\processed\qa_results_fixed_embedding.csv (22행)
컬럼: instance, size, formulation, linking, qubo_variables, qubo_quadratic_terms, solver, status, runtime, num_reads, best_energy, best_objective, best_is_feasible, best_total_violation, best_max_violation, feasible_fraction, best_feasible_objective, best_feasible_energy, num_samples, energy_mismatch, reported_best_energy, argmin_agreement, unique_samples, feasible_reads, qa_chain_strength, qa_chain_strength_alpha, qa_annealing_time, qa_chain_break_method, qa_chain_break_fraction, qa_wall_clock, qa_qpu_access_time_us, qa_qpu_sampling_time_us, embedding_status, logical_variables, physical_qubits, max_chain_length, mean_chain_length, embedding_search_time, qa_message, seed, selected_by, solver_id


,seed,num_reads,qa_annealing_time,qa_chain_strength_alpha,feasible_fraction,qa_chain_break_fraction,runtime,qa_qpu_access_time_us,qa_qpu_sampling_time_us
0,2024,1000,20.0,1.5,0.0,0.021483,2.222270,226839.16,210920.0
1,2026,1000,20.0,1.5,0.0,0.018463,3.206760,228320.36,212400.0
2,2024,1000,50.0,1.5,0.0,0.018181,3.180810,256839.16,240920.0
3,2026,1000,50.0,1.5,0.0,0.016278,2.225688,258320.36,242400.0
4,2024,1000,100.0,1.5,0.0,0.016707,2.115885,306839.16,290920.0
5,2026,1000,100.0,1.5,0.0,0.017224,2.316912,308320.36,292400.0
6,2024,1000,120.0,1.5,0.0,0.017749,2.187855,326839.16,310920.0
7,2026,1000,120.0,1.5,0.0,0.017228,3.317463,328320.36,312400.0
8,2024,1000,150.0,1.5,0.0,0.016502,2.324168,356839.16,340920.0
9,2026,1000,150.0,1.5,0.0,0.016506,2.440749,358320.36,342400.0


## embedding 품질과 QA 결과

물리 큐빗이 적고 chain이 짧은 embedding이 실제로 더 나은 결과를 내는지 확인합니다.

In [13]:
FQ.summarize(merged)

,seed,selected_by,physical_qubits,max_chain_length,qa_chain_break_fraction,feasible_fraction,best_feasible_objective,status
0,2024,"physical_qubits, avg_chain_length",4503,34,0.021483,0.0,None,OK
1,2026,max_chain_length,4537,31,0.018463,0.0,None,OK


## 4x4와 비교 — 이 notebook의 핵심

chain 길이가 크게 다른 두 경우를 나란히 놓습니다.

- **6x6도 chain break가 낮은데 feasible이 0** 이면, chain은 병목이 아니고 coefficient range가 지배적이라는 뜻입니다.
- **6x6에서 chain break가 급증** 했다면 chain 길이도 유의한 요인입니다.

4x4 결과는 notebook 05의 QA 결과에서 가져옵니다.

In [ ]:
# 주의: qa_results.csv는 chain 평균을 mean_chain_length로,
#       embedding_study.csv는 avg_chain_length로 쓴다. 두 모듈을 따로
#       만들면서 생긴 불일치인데, 지금 통일하면 이미 쌓인 CSV가 깨지므로
#       여기서 각각의 이름을 맞춰 읽는다.
try:
    qa_prev = load_table(RAW_DIR, "qa_results.csv")
    prev = qa_prev[
        (qa_prev["instance"] == "4x4")
        & (qa_prev["formulation"] == "MS")
    ]
    # linking 두 변형으로 돌렸다면 본 실험 설정(linking 제외)만 본다.
    if "linking" in prev.columns:
        prev = prev[~prev["linking"].astype(bool)]
    columns = [c for c in (
        "instance", "formulation", "linking", "status",
        "embedding_status", "physical_qubits", "max_chain_length",
        "mean_chain_length", "qa_chain_break_fraction",
        "feasible_fraction",
    ) if c in prev.columns]
    print("[notebook 05] MS 4x4 (linking 제외)")
    display(prev[columns])
    if "embedding_status" in prev.columns:
        status = set(prev["embedding_status"].dropna().unique())
        if status and status != {"OK"}:
            print(
                f"경고: embedding_status={status}. QPU 실행 기록이 "
                f"아니면 chain break 비교가 성립하지 않습니다. "
                f"아래 셀로 4x4를 같은 경로에서 다시 돌리십시오."
            )
except FileNotFoundError:
    print("notebook 05의 qa_results.csv가 없습니다.")

print("\n[이번 실행] MS 6x6")
display(merged[[
    "seed", "physical_qubits", "max_chain_length",
    "avg_chain_length", "qa_chain_break_fraction",
    "feasible_fraction", "true_gap_percent",
]])

## (선택) 4x4를 같은 경로로 다시 돌리기

위 4x4 기록은 notebook 05에서 **다른 코드 경로**로 얻은 것입니다. `TARGET_INSTANCE`를 `"4x4"`로 바꾸고 이 notebook을 처음부터 다시 실행하면, embedding 품질만 다르고 **나머지 조건이 완전히 동일한 대조**가 됩니다.

그러면 chain 길이가 chain break와 feasibility에 미치는 영향을 훨씬 깨끗하게 볼 수 있습니다. 4x4는 QPU 시간도 얼마 들지 않습니다.

위 셀에서 `embedding_status`가 `OK`가 아니라는 경고가 떴다면 **반드시** 이 방법으로 다시 돌리십시오. 토큰 없이 기록된 행이면 chain break 값 자체가 없습니다.

## 해석 메모

아래는 결과를 보고 직접 채우십시오. 자동 판정은 두지 않았습니다. 표본이 seed 2~3개뿐이라 통계적 결론을 내리기에 부족하고, 잘못된 자동 판정이 오히려 오해를 만들 수 있기 때문입니다.

확인할 것

1. chain break 평균이 4x4(0.025%)에 비해 얼마나 올랐는가
2. feasible 비율이 0을 넘겼는가
3. embedding 품질(큐빗 수, 최대 chain)과 chain break에 상관이 보이는가
4. gap이 나왔다면 SA 결과(6x6 MS, 30.9%)와 비교해 어느 정도인가

**주의**: alpha를 바꿔 다시 돌린 결과는 본 실험이 아니라 진단 실험으로 분류해 기록하십시오. 본 실험의 QA 설정은 alpha=1.5 고정입니다.